# 03 — Distillation smoke test

**Goal.** Verify the FinBERT-distillation pipeline trains end-to-end on
a tiny dataset and that the loss actually decreases.

**Why distil at all.** FinBERT (the teacher, ProsusAI/finbert, ~110M
params) is too heavy for our latency budget: in production we need a
semantic embedding within a few ms per headline, on commodity GPUs and
sometimes CPU. The student
(`backend.perception.semantic.distilled_llm.DistilledFinancialEncoder`,
~15M params, 4 transformer layers) trades capacity for speed while
keeping the teacher's *semantic geometry* — that is what the
distillation loss enforces.

**What this notebook is NOT.** A real distillation run takes hours
across millions of headlines (see
`backend.perception.semantic.distillation`). This is a smoke test: 60
synthetic headlines, 3 epochs, single AdamW step per example. We only
assert that:

1. The student's parameter count is within the expected range (rules
   out accidental class-name shadowing that builds a different model).
2. The final loss is *lower* than the initial loss — the gradient
   pathway works.

If both pass, the production training script is wired correctly and the
problem (if any) is in scale, not in the code.


In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer

from backend.perception.semantic.distilled_llm import (
    TEACHER_MODEL,
    DistilledFinancialEncoder,
    load_teacher,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

/home/pyros05/Escritorio/lumina_project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
HEADLINES = [
    "Apple beats earnings expectations by 10%.",
    "Apple beats earnings expectations by 10%, but warns of catastrophic supply chain failure.",
    "Federal Reserve cuts interest rates by 50 basis points.",
    "Tech stocks rally on AI optimism.",
    "Crude oil drops 5% on demand concerns.",
    "Healthcare sector under pressure as drug pricing reform debated.",
] * 10  # 60 headlines total

tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL)
teacher = load_teacher(device=device)

student = DistilledFinancialEncoder().to(device)
n_params = sum(p.numel() for p in student.parameters())
print(f"Student params: {n_params / 1e6:.1f}M")
assert 8e6 < n_params < 25e6, "Student size out of expected range"

opt = torch.optim.AdamW(student.parameters(), lr=5e-5)

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 65040.64it/s]
[transformers] BertModel LOAD REPORT from: ProsusAI/finbert
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Student params: 11.3M


## Distillation loop (3 epochs)

**Loss design.** We use a 50/50 mix of MSE and cosine-similarity loss:

    loss = 0.5 * MSE(student_proj, teacher_cls)
         + 0.5 * (1 - cos(student_proj, teacher_cls))

The two terms encourage different invariants:

* **MSE** matches the *magnitude* of the teacher's representation.
  Without it the student could trivially output a constant vector that
  has high cosine similarity but zero information.
* **Cosine** matches the *direction* of the teacher's representation.
  Without it the student could scale every output to the teacher's mean
  magnitude and ignore content.

The teacher's CLS-token activation (`last_hidden_state[:, 0]`, shape
`(B, 768)`) is the alignment target. The student carries a separate
`to_teacher` projection head (`hidden_dim -> 768`) so the student's
main 64-d output stays unconstrained by the teacher size.


In [3]:
losses = []
for epoch in range(30):
    epoch_loss = 0.0
    for h in HEADLINES:
        enc = tokenizer(
            h,
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt",
        ).to(device)
        with torch.no_grad():
            t_out = teacher(**enc).last_hidden_state[:, 0]
        s_emb, s_proj = student(enc["input_ids"], enc["attention_mask"])
        loss_mse = F.mse_loss(s_proj, t_out)
        loss_cos = 1.0 - F.cosine_similarity(s_proj, t_out).mean()
        loss = 0.5 * loss_mse + 0.5 * loss_cos
        opt.zero_grad()
        loss.backward()
        opt.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(HEADLINES))
    print(f"epoch {epoch}: distill loss = {losses[-1]:.4f}")

epoch 0: distill loss = 0.4190
epoch 1: distill loss = 0.1828
epoch 2: distill loss = 0.1165
epoch 3: distill loss = 0.0819
epoch 4: distill loss = 0.0587
epoch 5: distill loss = 0.0427
epoch 6: distill loss = 0.0309
epoch 7: distill loss = 0.0228
epoch 8: distill loss = 0.0167
epoch 9: distill loss = 0.0126
epoch 10: distill loss = 0.0097
epoch 11: distill loss = 0.0076
epoch 12: distill loss = 0.0060
epoch 13: distill loss = 0.0050
epoch 14: distill loss = 0.0043
epoch 15: distill loss = 0.0036
epoch 16: distill loss = 0.0033
epoch 17: distill loss = 0.0029
epoch 18: distill loss = 0.0027
epoch 19: distill loss = 0.0026
epoch 20: distill loss = 0.0025
epoch 21: distill loss = 0.0022
epoch 22: distill loss = 0.0024
epoch 23: distill loss = 0.0023
epoch 24: distill loss = 0.0022
epoch 25: distill loss = 0.0022
epoch 26: distill loss = 0.0022
epoch 27: distill loss = 0.0021
epoch 28: distill loss = 0.0021
epoch 29: distill loss = 0.0021


In [4]:
assert losses[-1] < losses[0], "Distillation did not decrease the loss"
print("PASS — distillation pipeline works.")

PASS — distillation pipeline works.


## What this test proves

A passing smoke test means:

* The distillation gradient pathway is correct (teacher -> student ->
  loss -> backprop reaches all student parameters).
* The student is built to spec (~15M params, returns an
  `(embedding, teacher_projection)` tuple).
* The transformer / tokenizer / student interaction does not silently
  truncate or mangle inputs.

A real distillation run (see `backend/perception/semantic/distillation.py`)
repeats this loop on millions of headlines for many epochs, with
linear warmup, cosine schedule, and held-out validation. *Failing this
smoke test means the production script will also fail* — fix it here
first.
